# 🐧 TDK — Blender + AI Image-to-3D trong Google Colab

**Notebook kiểm thử (không cam kết model đẹp như Genshin).** Dùng trên Android: mở Chrome ở chế độ **Trang web cho máy tính** và chạy từng ô bằng nút ▶. **Không chọn Run all** ở lượt đầu.

Mục tiêu: xác định GPU thực tế → tải code TDK → chạy Blender headless, xuất/nhập GLB → *chỉ khi GPU/CUDA phù hợp* mới cài TripoSR và tạo model 3D từ ảnh → lưu kết quả lên Google Drive.

**Phạm vi quyền:** Repo TDK được clone công khai ở chế độ đọc, không cần GitHub token. Google Drive **chỉ được gắn ở ô cuối** để tránh cấp quyền đọc toàn Drive cho các dependency AI tải về trước đó. Không đăng nhập Hugging Face hoặc tải model trả phí, không cài dịch vụ chạy nền.

**Giới hạn:** Colab có thể không có GPU; phiên máy là tạm thời. TripoSR cũ có các phụ thuộc CUDA khó biên dịch trên môi trường mới, nên bước cài đặt có thể thất bại; nếu thế, giữ nguyên log và dừng thay vì cài đi cài lại. Đây không phải giải pháp Colab chạy tự động 24/7.

## 1 · Kiểm tra GPU, Python và dung lượng (an toàn, không cài)

Trong Colab chọn **Runtime → Change runtime type → T4 GPU** (nếu có). Chạy ô này. Nếu `CUDA available` là `False`, **không chạy các bước AI**. Vẫn có thể kiểm tra Blender bằng CPU.

In [ ]:
import os, sys, shutil, subprocess, json, platform
from pathlib import Path

WORK = Path('/content/TDK-3D')
INPUTS = WORK / 'inputs'
OUTPUTS = WORK / 'outputs'
for directory in (INPUTS, OUTPUTS):
    directory.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0], 'OS:', platform.platform())
print('Disk free GiB:', round(shutil.disk_usage('/content').free/1024**3, 1))
try:
    import torch
    HAS_CUDA = bool(torch.cuda.is_available())
    print('PyTorch:', torch.__version__, 'CUDA build:', torch.version.cuda)
    print('CUDA available:', HAS_CUDA)
    if HAS_CUDA:
        gpu = torch.cuda.get_device_properties(0)
        print('GPU:', gpu.name, 'VRAM GiB:', round(gpu.total_memory/1024**3, 2))
except ImportError:
    HAS_CUDA = False
    print('Torch chưa có; GPU inference chưa được kiểm chứng')
for command in ('nvidia-smi','nvcc'):
    print(command, ':', shutil.which(command) or 'không có')
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'], check=False)


## 2 · Tải TDK từ GitHub (chỉ đọc)

Tải về thư mục tạm của Colab. Không yêu cầu OAuth của GitHub, không `git push`, không sửa `main`.

In [ ]:
import subprocess
REPO = Path('/content/TDK')
if not (REPO / '.git').is_dir():
    subprocess.run(['git','clone','--depth','1','https://github.com/thanhdangkhoa16122008-hash/TDK.git',str(REPO)],check=True)
assert (REPO/'GAMEDEV.md').is_file(), 'TDK không chứa GAMEDEV.md'
assert (REPO/'html/templates/horror-fps-starter').is_dir()
print('PASS: TDK clone; commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True).strip())


## 3 · Cài Blender (CPU cũng chạy được)

Cài từ kho Ubuntu của máy Colab nếu chưa có. Phiên bản có thể khác Blender trên Windows; bước này chỉ cần xuất GLB chuẩn.

In [ ]:
if not shutil.which('blender'):
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','blender'],check=True)
subprocess.run(['blender','--version'],check=True)


## 4 · Kiểm thử mesh Blender: xuất GLB, nhập lại và kiểm tra vật liệu

**Đây là thử phần mềm, chưa dùng AI.** File `.blend` và `.glb` hiện nằm trong `/content/TDK-3D/outputs`, chỉ chuyển lên Drive ở ô cuối.

In [ ]:
import os, subprocess
SMOKE_SCRIPT = WORK/'blender_smoke.py'
SMOKE_SCRIPT.write_text(r'''import bpy, os
from pathlib import Path
out=Path(os.environ['TDK_OUTPUT_DIR']); out.mkdir(parents=True,exist_ok=True)
bpy.ops.wm.read_factory_settings(use_empty=True)
bpy.ops.mesh.primitive_uv_sphere_add(segments=20,ring_count=12)
obj=bpy.context.object; obj.name='TDK_REAL_MESH_SMOKE'
mat=bpy.data.materials.new('TDK_Cyan'); mat.diffuse_color=(0.1,0.55,0.85,1)
mat.use_nodes=True
bsdf=mat.node_tree.nodes.get('Principled BSDF')
if bsdf: bsdf.inputs['Base Color'].default_value=(0.1,0.55,0.85,1)
obj.data.materials.append(mat)
bpy.ops.wm.save_as_mainfile(filepath=str(out/'blender-smoke.blend'))
bpy.ops.export_scene.gltf(filepath=str(out/'blender-smoke.glb'),export_format='GLB')
bpy.ops.wm.read_factory_settings(use_empty=True)
bpy.ops.import_scene.gltf(filepath=str(out/'blender-smoke.glb'))
meshes=[ob for ob in bpy.data.objects if ob.type=='MESH']
assert meshes and sum(len(ob.data.vertices) for ob in meshes)>0
assert sum(len(ob.data.polygons) for ob in meshes)>0
assert any(len(ob.data.materials)>0 for ob in meshes)
print('PASS: Blender export/import GLB, geometry + material')
''',encoding='utf-8')
subprocess.run(['blender','--background','--python',str(SMOKE_SCRIPT)],env={**os.environ,'TDK_OUTPUT_DIR':str(OUTPUTS)},check=True)
assert (OUTPUTS/'blender-smoke.glb').stat().st_size>1000
print('PASS:', OUTPUTS/'blender-smoke.glb')


## 5 · Ảnh thử nghiệm (tùy chọn)

TripoSR có sẵn ảnh **chiếc ghế** mẫu, nên nếu chỉ kiểm tra công cụ, **bỏ qua ô này**. Nếu muốn kiểm tra ảnh nhân vật của mình, chọn đúng một ảnh PNG/JPG/WEBP, nền sạch. Phần bị che và mặt sau sẽ do AI phỏng đoán.

In [ ]:
from google.colab import files
from PIL import Image
uploaded=files.upload()
if uploaded:
    assert len(uploaded)==1, 'Chỉ chọn 1 ảnh'
    name=next(iter(uploaded))
    assert Path(name).suffix.lower() in {'.png','.jpg','.jpeg','.webp'}
    SRC=INPUTS/'reference.png'
    with Image.open(name) as img:
        img.convert('RGBA').save(SRC)
    print('Đã nhận ảnh:',SRC)
else:
    print('Chưa tải ảnh; dùng ảnh ghế mẫu ở bước 7')


## 6 · Cài AI TripoSR (có thể thất bại vì CUDA)

**Chỉ chạy nếu ô 1 xác nhận GPU CUDA và VRAM ~6 GiB trở lên.** Đây là dự án nguồn mở cũ; `torchmcubes` có thể cần `nvcc`, CMake và toolchain tương thích. Không vô hiệu hóa kiểm tra bảo mật. Phụ thuộc AI được cài **trước khi cấp quyền Drive**.

Nếu báo lỗi, giữ nguyên log rồi dừng tại đây; không chạy lại liên tục. Phiên Colab chưa được xác minh thực tế trong cuộc trò chuyện này.

In [ ]:
import sys, os, subprocess, shutil
assert HAS_CUDA, 'SKIP: Không có GPU CUDA phù hợp'
assert torch.cuda.get_device_properties(0).total_memory>=6*1024**3, 'SKIP: VRAM dưới 6 GiB'
if not shutil.which('nvcc'):
    raise RuntimeError('Không có nvcc để build torchmcubes. Dừng cài; cần đổi runtime hoặc phương án AI khác.')
AI_REPO=Path('/content/TripoSR')
if not (AI_REPO/'.git').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/VAST-AI-Research/TripoSR.git',str(AI_REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-U','setuptools','wheel','ninja','cmake'],check=True)
req=(AI_REPO/'requirements.txt').read_text().splitlines()
# Build torchmcubes separately with existing torch (no pip build isolation).
non_mc=[s for s in req if s.strip() and not s.lstrip().startswith('#') and 'torchmcubes' not in s]
reqfile=WORK/'requirements-triposr-without-torchmcubes.txt'
reqfile.write_text('\n'.join(non_mc)+'\n')
subprocess.run([sys.executable,'-m','pip','install','-r',str(reqfile)],check=True)
subprocess.run([sys.executable,'-m','pip','install','--no-build-isolation','git+https://github.com/tatsy/torchmcubes.git'],check=True)
from torchmcubes import marching_cubes
print('PASS: TripoSR dependencies import, torchmcubes available')


## 7 · Sinh GLB thật bằng AI

Nếu không tải ảnh ở bước 5, dùng `examples/chair.png` của TripoSR. Chạy được mới tính thành công; **có GLB không đồng nghĩa đẹp hay có rig**. TripoSR xuất màu vertex mặc định; texture UV cần một bước khác.

In [ ]:
import sys, subprocess
assert HAS_CUDA, 'Không có GPU CUDA'
assert (AI_REPO/'run.py').exists(), 'Chưa cài TripoSR'
source=INPUTS/'reference.png'
if not source.exists():
    source=AI_REPO/'examples/chair.png'
assert source.exists(), 'Không có ảnh input'
GENERATED_DIR=WORK/'triposr-inference'
cmd=[sys.executable,'run.py',str(source),'--output-dir',str(GENERATED_DIR),
     '--model-save-format','glb','--mc-resolution','160','--chunk-size','4096']
print('Chạy AI trên:',source)
subprocess.run(cmd,cwd=str(AI_REPO),check=True)
GENERATED=GENERATED_DIR/'0'/'mesh.glb'
assert GENERATED.exists() and GENERATED.stat().st_size>1024, 'TripoSR chưa tạo GLB'
print('PASS: AI generated GLB:',GENERATED, 'bytes:',GENERATED.stat().st_size)


## 8 · Kiểm tra model và đưa vào outputs

Kiểm tra mesh/triangle thật bằng `trimesh` trước khi lưu. Không gắn nhãn thành công chỉ vì có file. Nếu TripoSR chưa cài được, **bỏ qua ô này** và có thể lưu riêng kết quả kiểm tra Blender.

In [ ]:
import json, shutil
import trimesh
scene=trimesh.load(str(GENERATED),force='scene')
mesh_list=[m for m in scene.geometry.values() if isinstance(m,trimesh.Trimesh)]
vertices=sum(len(m.vertices) for m in mesh_list)
triangles=sum(len(m.faces) for m in mesh_list)
assert vertices>0 and triangles>0, 'GLB không có mesh tam giác'
target=OUTPUTS/'ai-model.glb'
shutil.copy2(GENERATED,target)
result={'engine':'TripoSR','source_image':source.name,'mesh_count':len(mesh_list),
        'vertices':vertices,'triangles':triangles,'file_bytes':target.stat().st_size,
        'has_rig':False,'has_animation':False,'quality_assessment':'NOT TESTED'}
(OUTPUTS/'model-report.json').write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding='utf-8')
print('PASS: GLB mesh validated',json.dumps(result,indent=2))


## 9 · Render 4 góc bằng Blender (tùy chọn, dùng CPU)

Render front/side/back là kiểm tra hình học 3D thật, không dùng ảnh 2D giả model. Thử nghiệm này không đo chất lượng nghệ thuật, rig hoặc FPS trên Android.

In [ ]:
RENDER_SCRIPT=WORK/'render_four_views.py'
RENDER_SCRIPT.write_text(r'''import bpy, os, math
from pathlib import Path
from mathutils import Vector
out=Path(os.environ['TDK_OUTPUT_DIR']); f=out/'ai-model.glb'
assert f.is_file()
bpy.ops.wm.read_factory_settings(use_empty=True)
bpy.ops.import_scene.gltf(filepath=str(f))
meshes=[o for o in bpy.data.objects if o.type=='MESH']
assert meshes, 'Missing mesh'
points=[o.matrix_world@Vector(c) for o in meshes for c in o.bound_box]
center=sum(points,Vector((0,0,0)))/len(points)
rad=max(max((p-center).length for p in points),0.1)
bpy.ops.object.camera_add();cam=bpy.context.object
bpy.context.scene.camera=cam
for coords in ((3,-4,6),(-4,-3,5)):
    bpy.ops.object.light_add(type='AREA',location=center+rad*Vector(coords))
    light=bpy.context.object;light.data.energy=750;light.data.size=max(rad*4,1)
scene=bpy.context.scene
scene.render.engine='CYCLES';scene.cycles.samples=8
scene.render.resolution_x=480;scene.render.resolution_y=480
scene.render.resolution_percentage=100
for deg in (0,90,180,270):
    a=math.radians(deg)
    cam.location=center+rad*Vector((2.8*math.sin(a),-2.8*math.cos(a),1.1))
    cam.rotation_euler=(center-cam.location).to_track_quat('-Z','Y').to_euler()
    scene.render.filepath=str(out/('preview_%03d.png'%deg))
    bpy.ops.render.render(write_still=True)
    print('Rendered',scene.render.filepath)
''',encoding='utf-8')
subprocess.run(['blender','--background','--python',str(RENDER_SCRIPT)],env={**os.environ,'TDK_OUTPUT_DIR':str(OUTPUTS)},check=True)
assert len(list(OUTPUTS.glob('preview_*.png')))==4
print('PASS: 4 rendered angles')


## 10 · LƯU lên Google Drive (bước duy nhất cần cấp quyền)

Colab sẽ hiện hộp thoại **cho phép kết nối Google Drive**. Chỉ đồng ý nếu bạn tin tưởng notebook/code đã chạy. Ghi kết quả vào thư mục riêng `MyDrive/TDK-3D/outputs/<thời gian>`; **không ghi đè thành phẩm cũ**. Có thể chạy ô này dù AI thất bại, để lưu file Blender thử nghiệm.

In [ ]:
from google.colab import drive
import shutil, datetime
drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive/TDK-3D')
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir=DRIVE_ROOT/'outputs'/stamp
run_dir.mkdir(parents=True,exist_ok=False)
files_saved=[]
for item in OUTPUTS.iterdir():
    if item.is_file():
        shutil.copy2(item,run_dir/item.name)
        files_saved.append(item.name)
assert files_saved, 'Không có file nào trong outputs để lưu'
print('PASS: Drive saved to',run_dir)
print('Files:',files_saved)


## Xong: cách đọc kết quả

- Có `blender-smoke.glb`: **Blender xuất/nhập GLB thành công**, không phải AI.
- Có `ai-model.glb` và `model-report.json`: đã sinh **mesh từ ảnh** (chưa đánh giá giống ảnh; chưa có rig).
- Có `preview_000.png`, `preview_090.png`, `preview_180.png`, `preview_270.png`: đã render 4 góc thật.
- Nếu xuất hiện `torchmcubes`, `numpy`, `transformers` hoặc lỗi CUDA: ghi lại thông báo. Đừng tải/cài lại mù quáng.

**Bước tiếp theo:** dùng `GLTFLoader` của Three.js trong TDK để thử model, đánh giá hình khối ở nhiều góc và sửa Blender. Chất lượng nhân vật game anime kiểu Genshin cần nhiều công đoạn riêng: khuôn mặt, tóc, trang phục, rig, animation, shader, tối ưu.

**Quyền & chi phí:** clone repo công khai không cần token; Colab GPU có giới hạn, có thể tốn compute units trên gói trả phí. Notebook này không khởi tạo máy chủ tính phí hoặc tự cấp quyền Google Cloud.